In [19]:
import pandas as pd
import xlsxwriter as xw

data = pd.read_csv("D:\西安22运行数据\运行 22\预处理\C001951294036.csv")
data

,行程id,子行程id,车辆id,检测时间,经度,纬度,方向,星数,定位状态,gps速度,定位时间,时间戳,里程戳,油量戳,车速,转数,水温,电压,加速度(m/s^2),怠速状态
0,600052565820,351,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-08-07 12:41:35,108.899226,34.213235,0,6,1,0,2022-08-07 12:41:34,1.0,0,510,0.000000,970,48,13.6,0.000000,T
1,600052565820,351,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-08-07 12:41:35,108.899226,34.213235,0,6,1,0,2022-08-07 12:41:35,2.0,0,510,0.000000,970,48,13.6,0.000000,T
2,600052565820,351,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-08-07 12:41:35,108.899226,34.213235,0,6,1,0,2022-08-07 12:41:36,3.0,0,510,0.000000,970,48,13.6,0.000000,T
3,600052565820,351,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-08-07 12:41:35,108.899226,34.213235,0,6,1,0,2022-08-07 12:41:37,4.0,0,510,0.000000,970,48,13.6,0.000000,T
4,600052565820,351,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-08-07 12:41:35,108.899226,34.213235,0,6,1,0,2022-08-07 12:41:38,5.0,0,510,0.000000,970,48,13.6,0.000000,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48253,600056356144,765,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-09-23 16:37:40,108.909390,34.741548,5,12,1,116,2022-09-23 16:37:40,1791.0,46976,63710,32.222222,2196,90,13.4,-0.055556,F
48254,600056356144,765,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-09-23 16:37:40,108.909390,34.741548,5,12,1,116,2022-09-23 16:37:41,1792.0,46976,63710,32.222222,2196,90,13.4,-0.055556,F
48255,600056356144,765,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-09-23 16:37:40,108.909390,34.741548,5,12,1,116,2022-09-23 16:37:42,1793.0,46976,63710,32.222222,2196,90,13.4,-0.055556,F
48256,600056356144,765,e99a8cac-4849-4064-8dfa-6454e1dbbe8c,2022-09-23 16:37:40,108.909390,34.741548,5,12,1,116,2022-09-23 16:37:43,1794.0,46976,63710,32.222222,2196,90,13.4,-0.055556,F


In [18]:
#开始截取运动学片段
i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if (data.at[i,"子行程id"]==data.at[i+1,"子行程id"])and (data.at[i,"行程id"]==data.at[i+1,"行程id"]):
        if lock1 == 0  and (data.at[i+1,"时间戳"]-data.at[i,"时间戳"]==1) and(data.at[i,"车速"]==0)and (data.at[i,"转数"]!=0):
            s1 = i
            print(s1)
            lock1 = 1            
        if (lock1 == 1 and lock2 == 0) and (data.at[i,"车速"]!=0)and (data.at[i+1,"时间戳"]-data.at[i,"时间戳"]==1):
            s2 = i
            lock2 = 1 
            print(s2-1)
        if lock1 ==1 and lock2 ==1 and(data.at[i+1,"时间戳"]-data.at[i,"时间戳"]==1) and (data.at[i,"转数"]!=0):
            s3 = i
            print(s3-1)
            i-=2
            
            part = [s1,s2-1,s3-1]
            result.append(part)
            lock1 = 0
            lock2 = 0
print("总共的运动学片段：",len(result))

i = 0
while i < len(result):
    if result[i][2] - result[i][0] < 20 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于20秒的运动学片段：",len(result))            
result            


0


KeyboardInterrupt: 

In [ ]:
#开始截取运动学片段





#（1）如果相邻有效数据时间间隔超过 300s，则认为该运动学片段缺乏完整性，予以剔除。
#（2）如果某运动学片段中的怠速时段超过 300s，也认为该片段无效。
#（3）一个运动学片段的全部时长若小于 20s，则将该片段剔除。

（1）怠速状态：速度等于零并且发动机连续运转的过程；
（2）加速状态：速度不等于零并且加速度大于等于 0.15 m/s2 的连续运转过程；
（3）减速状态：速度不等于零并且加速度小于等于-0.15 m/s2 的连续运转过程；
（4）匀速状态：速度不等于零并且加速度的绝对值小于等于 0.15 m/s2 的连续运
转过程。


（1）加速行驶状态：车辆加速度大于0.15m/𝑠
2，且车速大于 0；
（2）减速行驶状态：车辆加速度小于−0.15m/𝑠
2，且车速大于 0；
（3）匀速行驶状态：车辆加速度大于−0.15m/𝑠
2，小于0.15m/𝑠2，且车速大于 0；
（4）怠速行驶状态：车辆已启动，但车速等于 0。

需要将片段 内 怠速 占 比 极高且低速行驶 的 部分 运动学 片 段予 以 剔
除


（2） 减速比例或者加速比例为 0，根据短行程的定义，必须包含加速段和减速段。

1）加速：当采样点的加速度大于 0.14m/s2 时，计为加速点;
（2）减速：当采样点的加速度小于-0.14m/s2 时，计为减速点;
（3）匀速：当采样点的加速度绝对值小于或等于 0.14m /s2，且速度大于或等于 1km/h
时，计为匀速点;
（4）怠速：当采样点的加速度绝对值小于 0.14m/s2，且速度小于 1km/h 时，计为怠
速点。




怠速是指
汽车发动机正常工作，车速度为零的状态。将加速、匀速、减速状态统称为运行状态。




加速工况：车辆行驶过程中的加速度 a≥0.14m/s2 的工况。
减速工况：车辆行驶过程中的加速度 a≤-0.14m/s2 的工况。
匀速工况：车辆行驶过程中的加速度 a 的绝对值小于 0.14 m/s2，即-0.14 
m/s2<a<0.14 m/s2，且车辆的行驶速度 v≥1km/h 的工况。
怠速工况：车辆行驶过程中的加速度 a 的绝对值小于 0.14 m/s2，即-0.14 
m/s2<a<0.14 m/s2，且车辆的行驶速度 v<1km/h 的工况。

（ １ ） 加速工况： 加速度＞ （ Ｕ５ ｍ／ｓ
２
且速度不为 ０ 的＾运转状态；
。 ） 磯工况； 加趙《■０ ． １ ５ ｍ／站速度不为 ０ 的織运转状态；
（ ３ ） 匀速工况： 加速度的绝对值《 ０． １ ５ ｍ／ｓ
２
且速度不为 ０ 的连续运转状态；
（ ４ ） 怠速工况： 发动祝工作且速度为 ０ 的连续运转过程。

In [284]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理0290024401260096.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进0290024401260096.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 146
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 109
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 88
开始写入表格


In [285]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理0290024574110560.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进0290024574110560.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 423
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 304
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 252
开始写入表格


In [286]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理0290026892654320.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进0290026892654320.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 396
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 287
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 224
开始写入表格


In [287]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理C001259137063.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进C001259137063.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 169
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 141
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 140
开始写入表格


In [288]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理AK12GU5S.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进AK12GU5S.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 0
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 0
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 0
开始写入表格


ValueError: Length mismatch: Expected axis has 0 elements, new values have 3 elements

In [ ]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理AK114BB9.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进AK114BB9.xlsx',index=None)

In [ ]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理AK12L6BB.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进AK12L6BB.xlsx',index=None)

In [289]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理0290023246981152.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进0290023246981152.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 4424
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 3558
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 3211
开始写入表格


In [290]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理C001653475180.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进C001653475180.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 3339
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 2631
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 2629
开始写入表格


In [280]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理33ACYGRY.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进33ACYGRY.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 1561
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 1342
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 766
开始写入表格


In [281]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理0290025900640180.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进0290025900640180.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 2176
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 1775
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 1559
开始写入表格


In [282]:
import pandas as pd
import xlsxwriter as xw

excel = pd.read_excel("D:\jupyter notebook 保存文件\已经预处理的文件\预处理C001781645168.xlsx")
data = pd.DataFrame(excel)
data = data.values.tolist()     # 转为列表

#截取运动学片段
print("开始截取运动学片段")

i = 0
s1= 0
s2= 0
s3= 0
lock1 = 0
lock2 = 0
result = []
garbage = []
while i < len(data):
    if lock1 == 0 and data[i][10]==0 and data[i][9]==0:  #速度和加速度都为0时为怠速
        s1 = i
        lock1 = 1
    if (lock1 == 1 and lock2 == 0 and (data[i][10] > 0.1))  or( lock1 == 1 and lock2 == 0 and (data[i][10]<-0.1)) or (lock1 == 1 and lock2 == 0 and (data[i][9]!=0 and data[i][10] > -0.1  and data[i][10]<0.1)):
        s2 = i
        lock2 = 1
    if lock1 ==1 and lock2 ==1 and data[i][10]==0 and data[i][9]==0:
        s3 = i
        i-=2
        
        part = [s1,s2-1,s3-1]
        result.append(part)
        lock1 = 0
        lock2 = 0
    
    # 存储相邻点时间差大于5秒的索引，然后删掉这个运动学片段
    if data[i][5] - data[i - 1][5] > 6:  # 大于6
        garbage.append(i)  # 放进去的这个点和前一个点的时间差大于5秒，存的是后一个点！
    i += 1
print("总共的运动学片段：",len(result))

#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))

#删掉时间小于10秒的运动学片段
print("开始删除持续时间小于10秒的运动学片段")
i = 0
while i < len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段三次改进C001781645168.xlsx',index=None)

开始截取运动学片段
总共的运动学片段： 775
开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 570
开始删除持续时间小于10秒的运动学片段
删除持续时间小于10秒的运动学片段： 563
开始写入表格


In [26]:
#删除缺失点过多的运动学片段
print("开始删除缺失点过多的运动学片段")

i = 0
while i < len(garbage):
    j = 0
    while j < len(result):
        if result[j][0] >= garbage[i]:   #开头大于、等于你，后面的元素就都不用判断了
            j+=1
            break
        if result[j][0] < garbage[i] <= result[j][2]:
            del result[j]  #少了一个元素，所以就不加1
        else:
            j+=1
    i+=1  # garbage长度不会变，每次循环完，正常+1就行
print("删除缺失点过多的运动学片段：",len(result))


开始删除缺失点过多的运动学片段
删除缺失点过多的运动学片段： 1572


In [30]:
#删掉时间小于10秒的运动学片段
i = 0
while i <len(result):
    if result[i][2] - result[i][1] < 10 :
        del result[i]
    else:
        i+=1
print("删除持续时间小于10秒的运动学片段：",len(result))

IndexError: list index out of range

In [28]:
print("开始写入表格")
from pandas.core.frame import DataFrame
data=DataFrame(result)
data.columns=["开始怠速",'怠速结束', '运动段结束']
data.to_excel('运动学片段二次改进21UEE92Q.xlsx',index=None)

开始写入表格
